# 🟩 Pattern 7 — Autonomous Goal-Seeking Agents

> **One-line definition:** you give a **goal**, not a task. The agent owns the loop until
> the goal is measurably met.

Ancestors: AutoGPT, BabyAGI. Modern form: **Deep Agents**, long-running research agents.

---

## 1. Mental Model

```
      GOAL (+ success criteria)
        │
        ▼
   ┌──────────┐
┌─►│ OBSERVE  │  what's the current world state?
│  └──────────┘
│        ▼
│  ┌──────────┐
│  │  PLAN    │  what should I do next, given the goal?
│  └──────────┘
│        ▼
│  ┌──────────┐
│  │   ACT    │  execute
│  └──────────┘
│        ▼
│  ┌──────────┐
│  │ EVALUATE │  ⭐ AM I DONE? measure against the criteria
│  └──────────┘
│        │
└────────┤ not yet
         │
         └── goal met / budget spent ──► DONE
```

---

## 2. What makes this different from everything before it

| | Patterns 1–6 | Pattern 7 |
|---|---|---|
| Human gives | a **task** | a **goal** |
| Loop owner | the human (turn by turn) | **the agent** |
| Stops when | the task is done | the **goal is measurably met** |
| Success is | implicit | ✅ **explicitly defined upfront** |
| Runs for | seconds | minutes → hours → days |
| Needs | — | **budgets, guardrails, HITL** |

👉 **The defining component is the EVALUATOR** — a node that answers "are we there yet?"
against criteria fixed *before* the run. Without it, you have a ReAct agent with a big
recursion limit, not an autonomous agent.

---

## 3. Key Properties (pointwise)

| Property | Autonomous agent |
|---|---|
| Planning | ✅ yes, and re-planning |
| Loop | ✅ agent-owned, long-running |
| Memory | ✅ required (it must survive restarts) |
| Reflection | ✅ the evaluator |
| Termination | goal met **or** budget exhausted |
| Cost | 💰💰💰 highest & unbounded — **budget it** |
| Risk | 🔴 highest — real side effects, no human in the loop |

---

## 4. The confusing parts (resolved 👇)

### Q1: "This is just ReAct with a while-loop, isn't it?"

The mechanics look similar. The **contract** is completely different:

| | ReAct | Autonomous |
|---|---|---|
| Stop signal | LLM emits no tool_calls | **evaluator** says criteria are met |
| Progress tracked? | ❌ no | ✅ yes, explicitly |
| Can it fail? | just returns something | ✅ **reports failure honestly** |
| Budget | recursion_limit (a crash) | ✅ a **first-class state field** |

An agent that can't say *"I could not achieve this goal"* is not autonomous — it's a
hallucination machine with extra steps.

---

### Q2: "Why not just let the LLM decide when it's done?"

Because **LLMs are optimists.** Ask "are you done?" and you get "yes!" far too early.

Fixes, strongest first:
1. 🥇 **Deterministic criteria** — `len(findings) >= 5 and all(f.has_source)`
2. 🥈 **Structured evaluator** with a numeric `confidence` and a threshold
3. 🥉 **Evaluator with tools** so it can verify claims
4. Always: a **hard budget** as backstop

---

### Q3: "What actually goes in state? It feels like a lot."

The taxonomy of an autonomous agent's state:

```python
class State(TypedDict):
    goal: str                  # WHAT — fixed, never changes
    success_criteria: List[str]# WHEN to stop — fixed, never changes
    plan: List[str]            # HOW — revised every cycle
    findings: List[str]        # WHAT WE LEARNED — accumulates (reducer!)
    iterations: int            # BUDGET — the safety rail
    max_iterations: int        # ↑
    goal_met: bool             # THE ANSWER
    final_report: str          # THE OUTPUT
```

**Rule:** `goal` and `success_criteria` are **immutable** — they're the anchor. Everything
that drifts (plan, findings) is separate. If the goal could be rewritten by the agent,
you've built something that always "succeeds".

---

### Q4: "How do I stop this thing burning $500 overnight?"

Layered defence. **All of them, not one.**

| Layer | Mechanism |
|---|---|
| 1. Iterations | `max_iterations` in state, checked in the router |
| 2. Graph steps | `config={"recursion_limit": N}` |
| 3. Tokens/cost | count in state, or use LangSmith limits |
| 4. Wall clock | store `started_at`, abort past a deadline |
| 5. Side effects | `interrupt_before=["dangerous_node"]` |
| 6. Tool scope | just don't give it `send_email` / `delete_*` |

---

### Q5: "Where does human-in-the-loop fit if it's 'autonomous'?"

Autonomy is a **spectrum, not a binary**:

```
supervised ──────────────────────────────► fully autonomous
approve every step   approve risky steps   report at the end
   (safe, slow)         ⭐ sweet spot        (fast, risky)
```

Most production "autonomous" agents sit in the middle: they loop freely on **read-only**
tools and interrupt for **write** actions.

---

## 5. What we'll build

- **Part A** — a full goal-seeking research agent (observe → plan → act → evaluate)
- **Part B** — budget guardrails
- **Part C** — HITL on dangerous actions


## 0. Setup

**Install once:**

```bash
pip install langgraph langchain-openai langchain-core
```

**Set your API key** (any chat model works — swap the import if you use Anthropic/Ollama).


In [ ]:
# --- Standard setup used by every notebook in this series ---
import getpass
import os


def _set(var: str):
    """Prompt for a key only if it's not already in the environment."""
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")


_set("GROQ_API_KEY")

from langchain_groq import ChatGroq

# temperature=0 -> deterministic-ish output, easier to reason about while learning
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("LLM ready")

---

# 🟦 PART A — The goal-seeking loop

## 6. Step 1 — Tools (read-only on purpose)

**Safety principle:** an autonomous agent should only get **read-only** tools by default.
Anything with side effects goes behind an interrupt (Part C).


In [ ]:
from langchain_core.tools import tool


@tool
def search(query: str) -> str:
    """Search a knowledge base for information about a topic."""
    kb = {
        "kafka": "Kafka: distributed log, ~1M msg/s per broker, p99 ~5ms, "
                 "durable via replication factor 3.",
        "rabbitmq": "RabbitMQ: AMQP broker, ~50k msg/s, richer routing, "
                    "weaker at replay/retention.",
        "pulsar": "Pulsar: segmented storage via BookKeeper, tiered offload, "
                  "~1M msg/s, native multi-tenancy.",
        "latency": "Latency: Kafka p99 5ms; RabbitMQ p99 1ms at low volume but "
                   "degrades sharply past 50k msg/s.",
        "durability": "Durability: Kafka replication + ISR; Pulsar quorum writes; "
                      "RabbitMQ mirrored queues (costly).",
    }
    hits = [v for k, v in kb.items() if k in query.lower()]
    return "\n".join(hits) if hits else f"No results for '{query}'."


@tool
def list_topics() -> str:
    """List every topic available in the knowledge base."""
    return "kafka, rabbitmq, pulsar, latency, durability"


tools = [search, list_topics]

---

## 7. Step 2 — State

⭐ Note which fields are **immutable** (the anchor) and which **accumulate** (need reducers).


In [ ]:
from typing import List, Annotated, TypedDict
import operator
from pydantic import BaseModel, Field


class AutoState(TypedDict):
    # === IMMUTABLE — the anchor. Never rewritten. ===
    goal: str
    success_criteria: List[str]

    # === MUTABLE — the work ===
    plan: List[str]  # replaced each cycle
    findings: Annotated[List[str], operator.add]  # ⭐ ACCUMULATES
    observations: str  # current world snapshot

    # === BUDGET — the safety rail ===
    iterations: int
    max_iterations: int

    # === OUTCOME ===
    goal_met: bool
    confidence: int
    final_report: str

---

## 8. Step 3 — OBSERVE

**Why a separate node?** So the plan is grounded in *what we actually have*, not in what the
LLM imagines we have. Cheap, deterministic, no LLM call.


In [ ]:
def observe(state: AutoState) -> dict:
    """Snapshot the current world state. Deterministic — no LLM, no cost."""
    findings = state.get("findings", [])
    obs = (
            f"Iteration: {state.get('iterations', 0)}/{state['max_iterations']}\n"
            f"Findings collected: {len(findings)}\n"
            + ("\n".join(f"  - {f[:120]}" for f in findings) if findings else "  (none yet)")
    )
    print(f"\n👁️  OBSERVE — {len(findings)} finding(s)")
    return {"observations": obs}

---

## 9. Step 4 — PLAN

⭐ The plan is regenerated **every cycle** from the *immutable goal* + *current observations*.
That's what keeps a long run from drifting.


In [ ]:
class NextActions(BaseModel):
    """Small batch of concrete next steps."""
    actions: List[str] = Field(
        description="1-3 concrete next actions. Each must be executable with the "
                    "available tools. Focus on the biggest remaining gap."
    )
    rationale: str = Field(description="One sentence: why these actions now?")


planner = llm.with_structured_output(NextActions)


def plan(state: AutoState) -> dict:
    """Re-plan from scratch each cycle, anchored on the immutable goal."""
    prompt = (
            f"GOAL: {state['goal']}\n\n"
            f"SUCCESS CRITERIA:\n" + "\n".join(f"- {c}" for c in state["success_criteria"]) +
            f"\n\nCURRENT STATE:\n{state['observations']}\n\n"
            f"Available tools: search(query), list_topics()\n\n"
            "What are the next 1-3 actions to close the biggest gap toward the goal? "
            "Do not repeat work already reflected in the findings."
    )
    result = planner.invoke(prompt)
    print(f"📋 PLAN: {result.rationale}")
    for a in result.actions:
        print(f"     • {a}")
    return {"plan": result.actions}  # no reducer -> REPLACES the old plan

---

## 10. Step 5 — ACT

A ReAct sub-agent again. **Pattern composition:** Autonomous = Plan + ReAct + Evaluate + Budget.


In [ ]:
from langgraph.prebuilt import create_react_agent

executor = create_react_agent(
    llm, tools=tools,
    prompt="Execute the given actions using the tools. Report ONLY factual findings. "
           "If a tool returns nothing useful, say so plainly — do NOT invent facts.",
)


def act(state: AutoState) -> dict:
    """Execute the planned actions; append what we learn to findings."""
    actions = "\n".join(f"{i}. {a}" for i, a in enumerate(state["plan"], 1))
    print("⚙️  ACT")
    res = executor.invoke({"messages": [("user", f"Execute these actions:\n{actions}")]})
    finding = res["messages"][-1].content
    print(f"     -> {finding[:140]}...")

    # operator.add appends. Also bump the iteration counter — the budget clock.
    return {"findings": [finding], "iterations": state.get("iterations", 0) + 1}

---

## 11. Step 6 — EVALUATE ⭐

**This node is the whole pattern.** Everything else is Patterns 2 + 3 recombined.

The evaluator judges **against the fixed criteria**, not against "does this look nice".


In [ ]:
class GoalCheck(BaseModel):
    """Structured verdict — makes 'done' a threshold, not an opinion."""
    goal_met: bool = Field(description="True ONLY if EVERY success criterion is satisfied")
    confidence: int = Field(description="Confidence 1-10 that the goal is genuinely met")
    unmet_criteria: List[str] = Field(description="Criteria still NOT satisfied. "
                                                  "Empty list if all are met.")
    reasoning: str = Field(description="One sentence justification.")


evaluator = llm.with_structured_output(GoalCheck)


def evaluate(state: AutoState) -> dict:
    """Am I done? Judged strictly against the IMMUTABLE criteria."""
    findings = "\n\n".join(state["findings"])
    prompt = (
            f"GOAL: {state['goal']}\n\n"
            f"SUCCESS CRITERIA:\n" + "\n".join(f"- {c}" for c in state["success_criteria"]) +
            f"\n\nEVIDENCE GATHERED:\n{findings}\n\n"
            "Be STRICT. Mark goal_met=True only if every single criterion is clearly "
            "satisfied by the evidence above. Partial coverage is NOT met."
    )
    check = evaluator.invoke(prompt)

    icon = "✅" if check.goal_met else "🔄"
    print(f"{icon} EVALUATE: met={check.goal_met} conf={check.confidence}/10 — {check.reasoning}")
    for u in check.unmet_criteria:
        print(f"     ✗ still missing: {u}")

    return {"goal_met": check.goal_met, "confidence": check.confidence}

---

## 12. Step 7 — The router (three exits, not two)

⭐ **The honest-failure exit is what separates this from a toy.**


In [ ]:
from langgraph.graph import StateGraph, START, END

CONFIDENCE_THRESHOLD = 7


def goal_router(state: AutoState) -> str:
    """Three outcomes: SUCCESS, BUDGET EXHAUSTED, or KEEP GOING."""

    # Exit 1 — SUCCESS. Both conditions: the verdict AND the confidence.
    if state["goal_met"] and state["confidence"] >= CONFIDENCE_THRESHOLD:
        print("🎯 Goal achieved")
        return "report"

    # Exit 2 — BUDGET EXHAUSTED. Report honestly; do NOT pretend success.
    if state["iterations"] >= state["max_iterations"]:
        print("⏹️  Budget exhausted — reporting partial results")
        return "report"

    # Otherwise: another cycle
    return "observe"

### The reporter — must be able to say "I failed"


In [ ]:
def report(state: AutoState) -> dict:
    """Produce the final output. Honesty about failure is mandatory."""
    status = ("GOAL ACHIEVED" if state["goal_met"]
              else "GOAL NOT FULLY ACHIEVED (budget exhausted)")
    findings = "\n\n".join(state["findings"])

    text = llm.invoke(
        f"GOAL: {state['goal']}\nSTATUS: {status}\n"
        f"Iterations used: {state['iterations']}/{state['max_iterations']}\n\n"
        f"EVIDENCE:\n{findings}\n\n"
        "Write the final report. If the goal was NOT achieved, say so explicitly at the "
        "top and list exactly what is still missing. Do not paper over gaps."
    ).content
    return {"final_report": text}

---

## 13. Step 8 — Assemble


In [ ]:
b = StateGraph(AutoState)

b.add_node("observe", observe)
b.add_node("plan", plan)
b.add_node("act", act)
b.add_node("evaluate", evaluate)
b.add_node("report", report)

b.add_edge(START, "observe")
b.add_edge("observe", "plan")
b.add_edge("plan", "act")
b.add_edge("act", "evaluate")

# ⭐ The autonomous loop + the two exits
b.add_conditional_edges("evaluate", goal_router, ["observe", "report"])
b.add_edge("report", END)

auto_graph = b.compile()
print(auto_graph.get_graph().draw_mermaid())

In [ ]:
from IPython.display import Image, display

try:
    display(Image(auto_graph.get_graph().draw_mermaid_png()))
except Exception:
    print("(rendering unavailable)")

---

## 14. Run it — give it a goal and walk away


In [ ]:
result = auto_graph.invoke(
    {
        "goal": "Determine whether Kafka, RabbitMQ or Pulsar is best for a "
                "high-throughput event backbone.",
        "success_criteria": [
            "Throughput figures for all three systems",
            "Latency characteristics for all three systems",
            "Durability model for all three systems",
            "A clear recommendation with justification",
        ],
        "findings": [],
        "iterations": 0,
        "max_iterations": 5,  # ⭐ the budget
        "goal_met": False,
        "confidence": 0,
    },
    config={"recursion_limit": 50},  # ⭐ the backstop
)

print("\n" + "=" * 60)
print(f"Iterations: {result['iterations']} | goal_met: {result['goal_met']} "
      f"| confidence: {result['confidence']}/10\n")
print(result["final_report"])

---

# 🟩 PART B — Budget guardrails

**Never ship an autonomous agent without these.** Cost is unbounded by construction.


In [ ]:
import time


class BudgetState(AutoState):
    started_at: float
    max_seconds: float
    tokens_used: int
    max_tokens: int


def budget_router(state: BudgetState) -> str:
    """Layered defence. Check EVERY limit, cheapest first."""

    # 1. Success
    if state["goal_met"] and state["confidence"] >= CONFIDENCE_THRESHOLD:
        return "report"

    # 2. Iteration budget
    if state["iterations"] >= state["max_iterations"]:
        print("⏹️  stop: iteration budget")
        return "report"

    # 3. Wall-clock budget (protects against a slow tool hanging the run)
    if time.time() - state["started_at"] > state["max_seconds"]:
        print("⏹️  stop: time budget")
        return "report"

    # 4. Token / cost budget
    if state["tokens_used"] >= state["max_tokens"]:
        print("⏹️  stop: token budget")
        return "report"

    return "observe"


print("""Guardrail layers — use ALL of them:
  1. max_iterations   in state  -> logical progress cap
  2. recursion_limit  in config -> graph-step backstop
  3. max_seconds      in state  -> wall-clock cap
  4. max_tokens       in state  -> cost cap
  5. interrupt_before           -> human gate on side effects
  6. tool selection             -> read-only by default
""")

---

# 🟦 PART C — Human-in-the-loop on dangerous actions

The realistic production shape: **loop freely on reads, pause on writes.**


In [ ]:
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


class ApprovalState(TypedDict):
    goal: str
    action: str
    approved: bool
    result: str


def propose(state: ApprovalState) -> dict:
    """Agent decides what it wants to do (read-only reasoning)."""
    return {"action": f"send_email(to='cto@corp.com', subject='{state['goal']}')"}


def human_gate(state: ApprovalState) -> dict:
    """⭐ interrupt() PAUSES the graph and surfaces a payload to the caller.
    Execution resumes only when you invoke with Command(resume=...)."""
    decision = interrupt({
        "question": "Approve this action?",
        "action": state["action"],
    })
    return {"approved": decision == "approve"}


def execute(state: ApprovalState) -> dict:
    if not state["approved"]:
        return {"result": "❌ Rejected by human — action NOT executed."}
    return {"result": f"✅ Executed: {state['action']}"}


ab = StateGraph(ApprovalState)
ab.add_node("propose", propose)
ab.add_node("human_gate", human_gate)
ab.add_node("execute", execute)
ab.add_edge(START, "propose")
ab.add_edge("propose", "human_gate")
ab.add_edge("human_gate", "execute")
ab.add_edge("execute", END)

# ⚠️ interrupt() REQUIRES a checkpointer — the pause must be persisted somewhere.
approval_graph = ab.compile(checkpointer=InMemorySaver())

In [ ]:
cfg = {"configurable": {"thread_id": "approval-1"}}

# 1. Run -> hits interrupt() and stops
out = approval_graph.invoke({"goal": "Q3 architecture review"}, cfg)
print("⏸️  PAUSED. Agent wants to:", out["__interrupt__"][0].value["action"])

# 2. Human decides. Command(resume=X) makes interrupt() RETURN X.
final = approval_graph.invoke(Command(resume="reject"), cfg)
print("\nResult:", final["result"])

In [ ]:
# Same graph, approved this time
cfg2 = {"configurable": {"thread_id": "approval-2"}}
approval_graph.invoke({"goal": "Q3 architecture review"}, cfg2)
final = approval_graph.invoke(Command(resume="approve"), cfg2)
print("Result:", final["result"])

---

## 15. Cheat Sheet

```
DEFINITION   human gives a GOAL; the agent owns the loop until measurably done
LOOP         observe -> plan -> act -> evaluate -> (loop | report)
DEFINING     ⭐ the EVALUATOR. Without it this is just ReAct with a big budget.
COMPOSITION  Autonomous = Planning + ReAct + Reflection + Memory + Budget

STATE RULES
  goal, success_criteria   IMMUTABLE  <- the anchor; never let the agent rewrite these
  plan                     replaced each cycle
  findings                 Annotated[list, operator.add]
  iterations/max           the safety rail
  goal_met, confidence     the verdict

THREE EXITS  success | budget exhausted (report honestly) | keep going
             ^ an agent that can't report failure is not autonomous

GUARDRAILS   iterations + recursion_limit + wall-clock + tokens + HITL + read-only tools
```

**API essentials**

| Task | Code |
|---|---|
| Structured verdict | `llm.with_structured_output(GoalCheck)` |
| Accumulate findings | `Annotated[List[str], operator.add]` |
| Loop | `add_conditional_edges("evaluate", router, ["observe","report"])` |
| Backstop | `config={"recursion_limit": 50}` |
| Pause | `interrupt({...})` (needs a checkpointer) |
| Resume | `graph.invoke(Command(resume=value), cfg)` |
| Static gate | `compile(interrupt_before=["execute"])` |

---

## 16. Failure modes

| Symptom | Cause | Fix |
|---|---|---|
| Declares success immediately | LLM optimism | strict evaluator + confidence threshold |
| Never terminates | criteria unachievable | `max_iterations` + honest failure report |
| Repeats the same action | plan ignores findings | feed `observations` into the planner |
| Burns budget on nothing | no progress detection | track findings-per-iteration; abort if flat |
| Hallucinated findings | no grounding | tools only; forbid inventing facts in the prompt |
| Costs explode | one guardrail only | layer all six |
| Did something irreversible | write tools + no gate | `interrupt_before` on every side effect |

---

## 17. Decision Tree

```
Who owns the loop?
├── The HUMAN (turn by turn) ──────► Patterns 1–6
└── The AGENT
    ├── Can success be defined BEFORE the run?
    │   └── NO ────────────────────► ⛔ don't build this. Use Planning + HITL instead.
    ├── Are there irreversible side effects?
    │   └── YES ───────────────────► ✅ AUTONOMOUS + interrupt_before (Part C)
    ├── Will it run for minutes+?
    │   └── YES ───────────────────► ✅ AUTONOMOUS + checkpointer + budgets (Part B)
    └── Read-only, bounded, well-defined goal?
        └── YES ───────────────────► ✅ AUTONOMOUS (Part A)
```

---

## 18. 🎓 The series in one picture

```
                        ┌─ Reactive (1) ─── state -> action, no loop
                        │
   add a CYCLE ─────────┼─ ReAct (2) ────── discover the path while acting
                        │
   add a PLAN ──────────┼─ Planning (3) ─── decide the path upfront, replan
                        │
   add a CRITIC ────────┼─ Reflective (4) ─ judge & revise the OUTPUT
                        │
   add PERSISTENCE ─────┼─ Memory (5) ───── identity across runs (augments 1-4,6,7)
                        │
   add more AGENTS ─────┼─ Multi-agent (6) ─ many decision makers
                        │
   add a GOAL + BUDGET ─┴─ Autonomous (7) ── the agent owns the loop
```

**Everything with a marketing name is a recombination:**

| "X agent" | = |
|---|---|
| Coding agent | Planning + Tools + Memory + **Self-Correction** |
| Research agent | ReAct/Planning + RAG + Tools + Memory |
| AI assistant | ReAct + Tools + **Memory** |
| Agentic RAG | ReAct/Planning + Retrieval |
| SWE agent | Planning + Reflection + Tools + exec sandbox |
| Deep research | **Autonomous** + Multi-agent + Reflection + Memory |

**The practical build order:**
```
1. Start with ReAct (2).                 It solves 70% of real problems.
2. Hitting quality issues?  → add Reflection (4).
3. Losing the thread?       → add Planning (3).
4. Forgetting the user?     → add Memory (5).
5. Too many tools?          → split into Multi-agent (6).
6. Needs to run alone?      → wrap in Autonomous (7) + guardrails.
```

⚠️ **Never start at 7.** Every step up multiplies cost, latency and debugging pain.
Earn each one.


In [22]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()  # load GROQ_API_KEY from .env if present

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.8)

prompt_template = PromptTemplate.from_template(
    f"You are a helpful assistant. Answer the following question:\n\n {{input}}"
)
(prompt_template | llm).invoke({"input": "Hello, world!"}).content

'Hello! 👋 How can I help you today?'